# MIMIC radiology sentences (proxy labels) → `ContrastiveMLP` ablations (Google Colab)

This notebook is the interactive version of `examples/mimic4_radiology_sentence_abnormality_contrastive_mlp.py`, adapted to run cleanly in **Google Colab**.

Prereqs:
- PhysioNet credentialed access + a copy of **MIMIC-IV Note** and **MIMIC-CXR** available to the Colab runtime (commonly via **Google Drive**)
- Run cells top-to-bottom


## Notes / limitations (read this)

- Uses **MIMIC-IV radiology** text + **MIMIC-CXR CheXpert** as a **patient-level proxy label** for sentence samples. This is **not** the paper’s GPT triple-vote teacher labeling.
- **Do not commit notebook outputs** containing raw report text to a public GitHub PR.

## Colab checklist

- Runtime → **GPU** (recommended)
- If your MIMIC files are on Drive: run the Drive mount cell, then verify `NOTE_ROOT` / `CXR_ROOT` exist
- If `set_task` is slow the first time: that is normal (LitData cache). Keep `CACHE_DIR` on local disk (`/tmp/...`) not Drive if Drive is slow


In [ ]:
# Colab-friendly PyHealth install
#
# - If you uploaded this notebook to Colab without the full repo, we clone PyHealth to /content/PyHealth.
# - If you are running inside a full git checkout, we install that checkout in editable mode.

import os
import sys
import subprocess
from pathlib import Path


def in_colab() -> bool:
    try:
        import google.colab  # type: ignore

        _ = google.colab
        return True
    except Exception:
        return False


def find_repo_root_with_pyproject(start: Path) -> Path | None:
    p = start.resolve()
    for _ in range(10):
        if (p / "pyproject.toml").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return None


# Default to Andrew's public fork for class/Colab reproducibility.
# Override with env vars if needed:
# - PYHEALTH_REPO_URL
# - PYHEALTH_REPO_BRANCH (optional)
PYHEALTH_REPO_URL = os.environ.get(
    "PYHEALTH_REPO_URL",
    "https://github.com/raymondcoding15/PyHealth.git",
)
PYHEALTH_REPO_BRANCH = os.environ.get("PYHEALTH_REPO_BRANCH", "").strip()

DEFAULT_COLAB_REPO = Path("/content/PyHealth")

if in_colab() and not find_repo_root_with_pyproject(Path.cwd()):
    if not DEFAULT_COLAB_REPO.exists():
        clone_cmd = ["git", "clone", "--depth", "1"]
        if PYHEALTH_REPO_BRANCH:
            clone_cmd += ["--branch", PYHEALTH_REPO_BRANCH]
        clone_cmd += [PYHEALTH_REPO_URL, str(DEFAULT_COLAB_REPO)]
        subprocess.check_call(clone_cmd)
    os.chdir(DEFAULT_COLAB_REPO)
    print("Using repo:", DEFAULT_COLAB_REPO)

REPO_ROOT = find_repo_root_with_pyproject(Path.cwd())
if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate pyproject.toml. In Colab, re-run after clone; locally, cd into PyHealth root."
    )

# Install editable package into the Colab kernel
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)])
print("Installed:", REPO_ROOT)


In [ ]:
# --- Configure your MIMIC roots ---
#
# Colab tip: put credentialed downloads on Google Drive and point to them here.
# Example layout:
#   /content/drive/MyDrive/mimic/mimic-iv-note/2.2/
#   /content/drive/MyDrive/mimic/mimic-cxr/2.0.0/

try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except Exception:
    # Local Jupyter: Drive mount not needed
    pass

# If you store both under one parent folder, you can set MIMIC_PARENT and keep defaults below.
MIMIC_PARENT = "/content/drive/MyDrive/mimic"  # change if needed

NOTE_ROOT = f"{MIMIC_PARENT}/mimic-iv-note/2.2"  # contains note/radiology.csv.gz
CXR_ROOT = f"{MIMIC_PARENT}/mimic-cxr/2.0.0"  # contains chexpert table files

# Speed/debug knobs
DEV = True  # limits patients for faster iteration (PyHealth dataset flag)
CACHE_DIR = "/tmp/pyhealth_litdata_cache"
epochs = 2


In [ ]:
from pyhealth.datasets import MIMIC4Dataset
from pyhealth.tasks.mimic4_radiology_sentence_chexpert_proxy import (
    MIMIC4RadiologySentenceCheXpertProxy,
)

base_dataset = MIMIC4Dataset(
    note_root=NOTE_ROOT,
    cxr_root=CXR_ROOT,
    note_tables=["radiology"],
    cxr_tables=["chexpert"],
    cache_dir=CACHE_DIR,
    dev=DEV,
    num_workers=1,
)

task = MIMIC4RadiologySentenceCheXpertProxy(bow_dim=128)
sample_dataset = base_dataset.set_task(task, num_workers=1)
print("samples:", len(sample_dataset))


In [ ]:
from __future__ import annotations

from typing import Dict

import torch

from pyhealth.datasets import get_dataloader, split_by_patient
from pyhealth.models import ContrastiveMLP
from pyhealth.trainer import Trainer


def run_ablation(sample_dataset, *, epochs: int) -> None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)

    settings = [
        {"hidden_dim": 64, "contrastive_weight": 0.0},
        {"hidden_dim": 64, "contrastive_weight": 0.1},
        {"hidden_dim": 128, "contrastive_weight": 0.1},
    ]

    for s in settings:
        train_ds, val_ds, _ = split_by_patient(sample_dataset, [0.7, 0.2, 0.1])
        train_loader = get_dataloader(train_ds, batch_size=32, shuffle=True)
        val_loader = get_dataloader(val_ds, batch_size=32, shuffle=False)

        model = ContrastiveMLP(dataset=sample_dataset, **s)
        trainer = Trainer(
            model=model,
            metrics=["accuracy", "roc_auc"],
            device=device,
            enable_logging=False,
        )
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader,
            epochs=epochs,
            monitor="roc_auc",
            optimizer_params={"lr": 1e-3},
        )
        metrics: Dict[str, float] = trainer.evaluate(val_loader)
        print(f"setting={s} -> {metrics}")


run_ablation(sample_dataset, epochs=epochs)


## Done

If this ran end-to-end, you now have:
- a `SampleDataset` built from real `radiology` text (sentence split)
- a trained `ContrastiveMLP` under multiple ablation settings
